# 03 — Target Tracking, Distractor Resilience, and Recovery Analysis

This notebook evaluates session-scoped target tracking, distractor spatial association, ambiguity detection, loss timeout, expiry, and reacquisition recovery for stage 03.

It imports reusable evaluation functions from `kinetiq_v_vision.evaluation.tracking` and asserts that the full scenario set — including an adversarial "Bystander-Only Reassociation" scenario — has zero silent target switches. `TargetTracker`'s spatial fallback (used when there is no direct candidate_id match) previously scored candidates by IoU/centroid distance only, with no identity check, so a same-position bystander could silently reach `CONFIRMED` under a different `candidate_id` once the true target left (VV-401). That fallback now never promotes a non-ID-matching candidate past `AMBIGUOUS` — identity can only be (re)established by the original `candidate_id` reappearing with plausible spatial continuity, or by an explicit `select_target` reconfirmation from the application layer. (The `bystander_attributed_reps` metric that used to appear here has been removed: repetition evaluation does not exist yet in this codebase, so that field was permanently and vacuously zero rather than measuring anything real.)

In [ ]:
from kinetiq_v_vision.evaluation.tracking import (
    create_synthetic_tracking_scenarios,
    run_tracking_evaluation,
)

print("Initializing Target Tracking Evaluation Stage 03...")

In [ ]:
scenarios = create_synthetic_tracking_scenarios()
print(f"Loaded {len(scenarios)} tracking evaluation scenarios:")
for s in scenarios:
    print(f" - {s.name}: {s.description} ({len(s.frames)} frames)")

In [ ]:
metrics = run_tracking_evaluation(scenarios)
print("\n--- Target Tracking Evaluation Summary ---")
print(f"Total Frames Evaluated:       {metrics.total_frames}")
print(f"Target Tracked Frames:        {metrics.target_tracked_frames}")
print(f"Attribution Accuracy:         {metrics.attribution_accuracy_pct}%")
print(f"Target Switch Errors:         {metrics.target_switch_count}")
print(f"False Pauses:                 {metrics.false_pause_count}")
print(f"Correct Pauses:               {metrics.correct_pause_count}")
print(f"Reacquisitions:               {metrics.reacquisition_count}")
print(f"Mean Reacquisition Latency:   {metrics.mean_reacquisition_latency_ms} ms")
print(f"Is Release Eligible:          {metrics.is_release_eligible}")
print(f"Is Synthetic Evaluation:      {metrics.is_synthetic}")

In [ ]:
# Programmatically enforce target tracking release invariants.
# The full scenario set -- including the adversarial "Bystander-Only
# Reassociation" scenario -- must now be switch-free and release eligible.
assert metrics.target_switch_count == 0, (
    f"Regression: {metrics.target_switch_count} silent target switch errors detected! "
    "TargetTracker's identity-continuity policy (process_frame steps 2-3) may have regressed."
)
assert metrics.is_release_eligible, "Target tracking metrics do not meet release eligibility requirements!"

print("Target tracking evaluation: all invariants verified, including the adversarial scenario.")